# ADL — Ethical Alignment Pipeline (v2)
**Reward Model + PPO/RLHF + ETHICS Evaluation**

## Datasets à monter sur Kaggle
| Slug | Contenu |
|------|---------|
| `adl-dpo-model-v3` (florianngo) | DPO adapter (LoRA) — chemin : `/kaggle/input/datasets/florianngo/adl-dpo-model-v3/dpo_model` |
| `adl-preferences` | `preferences.jsonl` (~10-20k paires) |
| `adl-reward-model-v2` *(optionnel)* | Reward model pré-entraîné pour skip step 3 |

**⚠ Toujours lancer toutes les cellules dans l'ordre depuis le début.**

In [ ]:
# ─── DOIT ÊTRE LA PREMIÈRE CELLULE — avant tout import torch ───
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CUDA_VISIBLE_DEVICES=", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
print("Installing dependencies...")
# bitsandbytes>=0.44.0 requis : corrige le bug 'frozenset has no attribute discard'
# avec transformers>=4.44 (validate_bnb_backend_availability)
!pip install -q -U \
    "bitsandbytes>=0.44.0" \
    "transformers==4.46.3" \
    "trl==0.12.0" \
    "peft==0.14.0" \
    "accelerate==1.2.0" \
    "datasets==3.2.0" \
    sentence-transformers \
    faiss-cpu \
    anthropic openai \
    "pyarrow==17.0.0" \
    tqdm

# Vérifie la version bitsandbytes installée
import importlib.metadata
bnb_ver = importlib.metadata.version("bitsandbytes")
print(f"✓ bitsandbytes {bnb_ver} (doit être >= 0.44.0)")
print("✓ Dependencies ready")

In [ ]:
import os, json, shutil, torch

# ── Chemins des datasets montés ──────────────────────────────────────────────
DPO_ADAPTER      = "/kaggle/input/datasets/florianngo/adl-dpo-model-v3/dpo_model"
PREFS_MOUNTED    = "/kaggle/input/adl-preferences/preferences.jsonl"
RM_MOUNTED       = "/kaggle/input/adl-reward-model-v2/reward_model"

BASE_MODEL       = "Qwen/Qwen2.5-1.5B-Instruct"

# ── Chemins de travail ───────────────────────────────────────────────────────
WORK             = "/kaggle/working/adl"
DATA_DIR         = f"{WORK}/data"
RESULTS_DIR      = f"{WORK}/results"
RM_DIR           = f"{RESULTS_DIR}/reward_model"
RLHF_DIR         = f"{RESULTS_DIR}/rlhf_model"
PREFS_PATH       = f"{DATA_DIR}/preferences.jsonl"
CORPUS_PATH      = f"{DATA_DIR}/ethical_corpus_v2.json"
EVAL_PATH        = f"{RESULTS_DIR}/eval_results.json"

# ── Vérification GPU ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"GPUs visibles: {torch.cuda.device_count()} (doit être 1)")
else:
    print("WARNING: No GPU!")

# ── Vérification DPO adapter ─────────────────────────────────────────────────
if os.path.isdir(DPO_ADAPTER):
    print(f"\n✓ DPO adapter: {DPO_ADAPTER}")
    print(f"  Contenu: {os.listdir(DPO_ADAPTER)[:5]}")
else:
    base_dpo = "/kaggle/input/datasets/florianngo/adl-dpo-model-v3"
    if os.path.isdir(base_dpo):
        print(f"DPO adapter pas trouvé à {DPO_ADAPTER}, contenu de la racine:")
        for root, dirs, files in os.walk(base_dpo):
            for f in files[:3]:
                print(f"  {os.path.join(root, f)}")
    else:
        print(f"✗ DPO adapter introuvable — monte le dataset adl-dpo-model-v3")

## 1 — Clone du repo & reset complet
> Cette cellule **efface tout** `/kaggle/working/adl` (code + résultats) pour repartir proprement.

In [ ]:
import os, shutil

WORK = "/kaggle/working/adl"

# Supprime TOUT (code + anciens résultats) pour éviter la contamination
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
    print(f"Répertoire {WORK} supprimé (reset propre)")

!git clone https://github.com/FeelTheFloww/adl-ethics.git /kaggle/working/adl

# Vérifie les scripts requis
REQUIRED = [
    "training/train_reward_model.py",
    "training/train_ppo.py",
    "evaluation/evaluate_ethics.py",
    "data/prepare_preferences.py",
]
print("\nScripts requis:")
all_ok = True
for rel in REQUIRED:
    path = f"/kaggle/working/adl/{rel}"
    ok = os.path.exists(path)
    print(f"  {'✓' if ok else '✗ MANQUANT — git push !':<30} {rel}")
    if not ok:
        all_ok = False

# Crée les répertoires de sortie (vides, propres)
for d in ["/kaggle/working/adl/data",
          "/kaggle/working/adl/results/reward_model",
          "/kaggle/working/adl/results/rlhf_model"]:
    os.makedirs(d, exist_ok=True)

if all_ok:
    print("\n✓ Tous les scripts présents — prêt à continuer")
else:
    print("\n⚠ Scripts manquants — faire git push depuis la machine locale avant de continuer")

## 2 — Données : `preferences.jsonl`

Priorité :
1. **Dataset monté** `adl-preferences` → recommandé, reproductible, rapide
2. Génération PKU-SafeRLHF + UltraFeedback en fallback (~20-30 min)

In [ ]:
import os, shutil, json

PREFS_MOUNTED = "/kaggle/input/adl-preferences/preferences.jsonl"
PREFS_PATH    = "/kaggle/working/adl/data/preferences.jsonl"
CORPUS_PATH   = "/kaggle/working/adl/data/ethical_corpus_v2.json"

# ── preferences.jsonl ────────────────────────────────────────────────────────
if os.path.exists(PREFS_MOUNTED):
    shutil.copy(PREFS_MOUNTED, PREFS_PATH)
    n = sum(1 for _ in open(PREFS_PATH))
    print(f"✓ preferences.jsonl copié depuis dataset monté : {n} paires")

elif os.path.exists(PREFS_PATH):
    n = sum(1 for _ in open(PREFS_PATH))
    print(f"✓ preferences.jsonl déjà présent : {n} paires")

else:
    print("preferences.jsonl absent — génération depuis PKU-SafeRLHF + UltraFeedback...")
    print("(~20-30 min — monte le dataset adl-preferences pour éviter ça)")
    !python /kaggle/working/adl/data/prepare_preferences.py \
        --n_pku   15000 \
        --n_ultra  5000 \
        --out_path /kaggle/working/adl/data/preferences.jsonl
    n = sum(1 for _ in open(PREFS_PATH))
    print(f"✓ Généré : {n} paires")

# ── Corpus éthique (RAG) ─────────────────────────────────────────────────────
for src in ["/kaggle/input/adl-corpus/ethical_corpus_v2.json",
            "/kaggle/working/adl/data/ethical_corpus_v2.json",
            "/kaggle/working/adl/data/ethical_corpus.json"]:
    if os.path.exists(src):
        if src != CORPUS_PATH:
            shutil.copy(src, CORPUS_PATH)
        print(f"✓ Corpus éthique prêt")
        break
else:
    print("⚠ Corpus éthique absent — RAG désactivé (baseline/dpo/rlhf OK)")

# ── Sanity check ─────────────────────────────────────────────────────────────
print("\nExemple de paires :")
with open(PREFS_PATH) as f:
    for i, line in enumerate(f):
        if i >= 2: break
        s = json.loads(line)
        print(f"  [{s.get('source','')}] {s['prompt'][:80]}...")

## 3 — Reward Model
Qwen2.5-1.5B-Instruct + LoRA (bfloat16) · Bradley-Terry loss · ~45-60 min sur T4

> Si `adl-reward-model-v2` est monté, la cellule suivante le copie et skip l'entraînement.

In [ ]:
import os, shutil

RM_DIR      = "/kaggle/working/adl/results/reward_model"
RM_MOUNTED  = "/kaggle/input/adl-reward-model-v2/reward_model"

SKIP_RM_TRAIN = False

# Le répertoire RM_DIR est vide après le clone (créé par makedirs sans contenu)
# Seul un adapter_config.json indique un vrai modèle entraîné
if os.path.isdir(RM_MOUNTED) and os.path.exists(f"{RM_MOUNTED}/adapter_config.json"):
    print(f"✓ Reward model monté trouvé — copie vers {RM_DIR}")
    shutil.rmtree(RM_DIR, ignore_errors=True)
    shutil.copytree(RM_MOUNTED, RM_DIR)
    SKIP_RM_TRAIN = True
    print("  → Entraînement reward model SKIPPÉ")
elif os.path.exists(f"{RM_DIR}/adapter_config.json"):
    print(f"✓ Reward model présent dans {RM_DIR} (entraîné dans cette session)")
    SKIP_RM_TRAIN = True
else:
    print(f"Reward model absent → entraînement requis")
    print(f"  (Pour skiper: monte adl-reward-model-v2)")

print(f"SKIP_RM_TRAIN = {SKIP_RM_TRAIN}")

In [ ]:
import os

RM_DIR = "/kaggle/working/adl/results/reward_model"

if not SKIP_RM_TRAIN:
    !python /kaggle/working/adl/training/train_reward_model.py \
        --model_name  Qwen/Qwen2.5-1.5B-Instruct \
        --data_path   /kaggle/working/adl/data/preferences.jsonl \
        --output_dir  /kaggle/working/adl/results/reward_model \
        --batch_size  1 \
        --grad_accum  16 \
        --max_length  384 \
        --lr          2e-5 \
        --epochs      1
else:
    print("Reward model déjà chargé — skip entraînement")

if os.path.exists(f"{RM_DIR}/adapter_config.json"):
    print(f"\n✓ Reward model prêt dans {RM_DIR}")
else:
    print(f"\n✗ adapter_config.json absent — l'entraînement a peut-être échoué")

In [ ]:
!zip -r /kaggle/working/reward_model_v4.zip /kaggle/working/adl/results/reward_model/
import os
size = os.path.getsize("/kaggle/working/reward_model_v4.zip") / 1e6
print(f"✓ reward_model_v4.zip ({size:.0f} MB) — sauvegarde via Output > New Dataset")

## 4 — PPO / RLHF
Policy warm-startée depuis le DPO adapter · KL-penalisé · TRL 0.12 API

**Mémoire** : 4 modèles sur T4 16 GB — `CUDA_VISIBLE_DEVICES=0` défini en cellule 1.

In [ ]:
!python /kaggle/working/adl/training/train_ppo.py \
    --model_name         Qwen/Qwen2.5-1.5B-Instruct \
    --dpo_adapter        /kaggle/input/datasets/florianngo/adl-dpo-model-v3/dpo_model \
    --reward_model_path  /kaggle/working/adl/results/reward_model \
    --data_path          /kaggle/working/adl/data/preferences.jsonl \
    --output_dir         /kaggle/working/adl/results/rlhf_model \
    --n_prompts          1500 \
    --total_episodes     1000 \
    --num_ppo_epochs     4 \
    --batch_size         2 \
    --grad_accum         4 \
    --lr                 1e-5 \
    --kl_coef            0.05 \
    --max_prompt_length  128 \
    --response_length    64 \
    --temperature        1.0

import os
RLHF_DIR = "/kaggle/working/adl/results/rlhf_model"
if os.path.exists(f"{RLHF_DIR}/adapter_config.json"):
    print(f"\n✓ RLHF model sauvegardé dans {RLHF_DIR}")
else:
    print(f"\n✗ adapter_config.json absent — vérifier la sortie ci-dessus")

In [ ]:
!zip -r /kaggle/working/rlhf_model_v2.zip /kaggle/working/adl/results/rlhf_model/
import os
size = os.path.getsize("/kaggle/working/rlhf_model_v2.zip") / 1e6
print(f"✓ rlhf_model_v2.zip ({size:.0f} MB) — sauvegarde via Output > New Dataset")

## 5 — Évaluation ETHICS

| Condition | Description |
|-----------|-------------|
| `baseline` | Qwen2.5-1.5B-Instruct brut |
| `dpo` | + fine-tuning DPO |
| `rlhf` | + fine-tuning PPO/RLHF |
| `rag` | baseline + retrieval de principes éthiques |

Catégories : *commonsense, deontology, justice, virtue, utilitarianism*

In [ ]:
!python /kaggle/working/adl/evaluation/evaluate_ethics.py \
    --base_model     Qwen/Qwen2.5-1.5B-Instruct \
    --dpo_adapter    /kaggle/input/datasets/florianngo/adl-dpo-model-v3/dpo_model \
    --rlhf_adapter   /kaggle/working/adl/results/rlhf_model \
    --ethical_corpus /kaggle/working/adl/data/ethical_corpus_v2.json \
    --output_path    /kaggle/working/adl/results/eval_results.json \
    --n_per_cat      100

## 6 — Résultats

In [ ]:
import json, os

EVAL_PATH = "/kaggle/working/adl/results/eval_results.json"

if not os.path.exists(EVAL_PATH):
    print("eval_results.json introuvable — lancer la cellule d'évaluation d'abord.")
else:
    with open(EVAL_PATH) as f:
        results = json.load(f)

    CATS = ["commonsense", "deontology", "justice", "virtue", "utilitarianism", "overall"]

    header = f"{'Condition':<22}" + "".join(f"  {c[:9]:>9}" for c in CATS)
    print(header)
    print("-" * len(header))

    for cond, res in results.items():
        row = f"{cond:<22}"
        for cat in CATS:
            acc = res.get(cat, {}).get("accuracy")
            row += f"  {acc:9.3f}" if acc is not None else f"  {'N/A':>9}"
        print(row)

    print()
    print("Meilleure condition par catégorie :")
    for cat in CATS:
        best_cond = max(
            results.keys(),
            key=lambda c: results[c].get(cat, {}).get("accuracy") or 0
        )
        best_acc = results[best_cond].get(cat, {}).get("accuracy", 0)
        print(f"  {cat:<18}: {best_cond} ({best_acc:.3f})")

## 7 — Sauvegarde finale

In [ ]:
import os, shutil

RLHF_DIR  = "/kaggle/working/adl/results/rlhf_model"
RM_DIR    = "/kaggle/working/adl/results/reward_model"
EVAL_PATH = "/kaggle/working/adl/results/eval_results.json"

for name, path in [("reward_model_final", RM_DIR), ("rlhf_model_final", RLHF_DIR)]:
    if os.path.isdir(path):
        out = f"/kaggle/working/{name}.zip"
        os.system(f"zip -r {out} {path}/")
        size = os.path.getsize(out) / 1e6
        print(f"  ✓ {name}.zip ({size:.0f} MB)")
    else:
        print(f"  ✗ {name}: répertoire absent ({path})")

if os.path.exists(EVAL_PATH):
    shutil.copy(EVAL_PATH, "/kaggle/working/eval_results.json")
    print("  ✓ eval_results.json")

print("\nFichiers dans /kaggle/working/:")
for f in sorted(os.listdir("/kaggle/working/")):
    try:
        size = os.path.getsize(f"/kaggle/working/{f}")
        print(f"  {f:40s} ({size/1e6:.1f} MB)")
    except:
        print(f"  {f}")

print("\nTout est prêt — Output > New Dataset pour sauvegarder.")